# Reusable Template: Multi-Factor Vulnerability Analysis with Public Demographic APIs

## Purpose
This notebook is a production-ready scaffold for identifying high-need geographic units (counties, tracts, etc.) using any public demographic API that returns tabular records.  
It follows the same structure used in the U.S. Census vulnerable-communities assignment:

1. Authenticate & retrieve data  
2. Parse → clean → type-cast  
3. Engineer rates & proportions  
4. Build a weighted composite vulnerability score  
5. Categorize with quantiles  
6. Flag statistical outliers  
7. Produce ranked tables and simple visualizations  

**How to adapt:**  
- Change `BASE_URL`, parameter names, and column lists.  
- Adjust the weight vector and the variables that enter the score.  
- Keep the overall pipeline intact so the notebook remains reproducible and auditable.

## 1. Imports & Configuration

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from dotenv import load_dotenv

# Optional helper (create your own or keep a simple min-max normalize)
def normalize(series: pd.Series) -> pd.Series:
    """Min-max scale a Series to [0, 1]."""
    return (series - series.min()) / (series.max() - series.min() + 1e-12)

# Visual defaults
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (8, 4)

## 2. Authenticate & Request Data

Store your API key in a `.env` file as `CENSUS_API_KEY=...` (or whatever name you choose).  
Never commit the key to version control.

In [ ]:
load_dotenv()

API_KEY = os.getenv("CENSUS_API_KEY")          # change env-var name if needed
BASE_URL = "https://2eraiuh.dlai.link/api/UScensus"  # replace with real endpoint

params = {
    "api_key": API_KEY,          # or "key" depending on the API contract
    # add any additional query parameters required by the endpoint
}

response = requests.get(BASE_URL, params=params)
print("Status code:", response.status_code)
assert response.status_code == 200, "API request failed – check key and URL"

In [ ]:
raw = response.json()
# Inspect structure once
print(type(raw))
if isinstance(raw, dict):
    print("Top-level keys:", list(raw.keys())[:10])

## 3. Convert to DataFrame

In [ ]:
# Adjust the key that holds the list of records
records = raw["data"] if isinstance(raw, dict) and "data" in raw else raw
df = pd.DataFrame(records)

print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
df.head()

## 4. Clean & Type-Cast

In [ ]:
# Drop incomplete rows (or implement a more sophisticated imputation strategy)
df_clean = df.dropna().copy()

# List every column that should be numeric
numeric_cols = [
    # "county", "state",
    # "population", "poverty_count",
    # "employed_total", "employed_male",
    # "male_pop_under_5", "female_pop_under_5",
    # "male_pop_over_75", "female_pop_over_75",
    # "poverty_count_male_under_5", "poverty_count_female_under_5",
    # "poverty_count_male_over_75", "poverty_count_female_over_75",
    # ... add / remove as needed for your API
]

for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").astype("Int64")

print("Remaining rows after dropna:", len(df_clean))
print(df_clean.dtypes)

## 5. Extract / Derive Geography Labels

In [ ]:
# Example: split "County Name, State" into separate columns
if "county_state" in df_clean.columns:
    split = df_clean["county_state"].str.split(", ", expand=True)
    df_clean["county_name"] = split[0]
    df_clean["state_name"] = split[1]

df_clean[["county_state", "state_name"]].head() if "state_name" in df_clean.columns else df_clean.head()

## 6. Feature Engineering – Rates & Proportions

Customize the formulas to the variables available from your API.

In [ ]:
# --- Total poverty rate ---
df_clean["total_poverty_rate"] = (
    df_clean["poverty_count"] / df_clean["population"]
)

# --- Under-5 poverty rate ---
poverty_under_5 = (
    df_clean["poverty_count_male_under_5"] + df_clean["poverty_count_female_under_5"]
)
pop_under_5 = (
    df_clean["male_pop_under_5"] + df_clean["female_pop_under_5"]
)
df_clean["under_5_poverty_rate"] = poverty_under_5 / pop_under_5

# --- Over-75 poverty rate ---
poverty_over_75 = (
    df_clean["poverty_count_male_over_75"] + df_clean["poverty_count_female_over_75"]
)
pop_over_75 = (
    df_clean["male_pop_over_75"] + df_clean["female_pop_over_75"]
)
df_clean["over_75_poverty_rate"] = poverty_over_75 / pop_over_75

# --- Employment rate ---
df_clean["employment_rate"] = df_clean["employed_total"] / df_clean["population"]

# --- Under-5 poverty as share of total poverty ---
df_clean["under_5_poverty_proportion"] = (
    (df_clean["poverty_count_male_under_5"] + df_clean["poverty_count_female_under_5"])
    / df_clean["poverty_count"]
)

df_clean[["total_poverty_rate", "under_5_poverty_rate", "over_75_poverty_rate"]].describe()

## 7. Composite Vulnerability Score

**Document your weights.**  
Default example mirrors the original assignment:  
- 35 % overall poverty  
- 40 % child-poverty concentration  
- 25 % unemployment

In [ ]:
W_POVERTY = 0.35
W_CHILD   = 0.40
W_UNEMP   = 0.25

raw_score = (
    df_clean["total_poverty_rate"] * W_POVERTY
    + df_clean["under_5_poverty_proportion"] * W_CHILD
    + (1 - df_clean["employment_rate"]) * W_UNEMP
)

df_clean["vulnerability_score"] = normalize(raw_score) * 100

sns.histplot(df_clean["vulnerability_score"], bins=30, kde=True)
plt.title("Vulnerability Score Distribution (0–100)")
plt.xlabel("Vulnerability Score")
plt.show()

## 8. Priority Categories (Quantile Binning)

In [ ]:
labels = ["Very Low", "Low", "Medium", "High", "Very High"]
df_clean["priority_score"] = pd.qcut(
    df_clean["vulnerability_score"],
    q=5,
    labels=labels
)

df_clean[["county_state", "vulnerability_score", "priority_score"]].head(10)

## 9. Ranked Tables & State Aggregation

In [ ]:
# Top 10 highest-need counties
top_counties = (
    df_clean
    .sort_values("vulnerability_score", ascending=False)
    [["county_state", "state_name", "vulnerability_score", "priority_score", "total_poverty_rate"]]
    .head(10)
)
print("Top 10 counties by vulnerability score:")
display(top_counties)

# States with the most "Very High" priority counties
very_high = df_clean[df_clean["priority_score"] == "Very High"]
state_counts = (
    very_high
    .groupby("state_name")
    .size()
    .sort_values(ascending=False)
    .reset_index(name="n_very_high_counties")
)
print("\nStates ranked by number of Very High priority counties:")
display(state_counts.head(15))

## 10. Statistical Outliers (Z-scores)

In [ ]:
col_for_outlier = "poverty_count"          # change to any absolute count of interest
Z_THRESHOLD = 3.0

df_clean["z_score"] = (
    (df_clean[col_for_outlier] - df_clean[col_for_outlier].mean())
    / df_clean[col_for_outlier].std()
)

outliers = df_clean[df_clean["z_score"] > Z_THRESHOLD].copy()
print(f"Counties with {col_for_outlier} z-score > {Z_THRESHOLD}: {len(outliers)}")
display(
    outliers
    .sort_values("z_score", ascending=False)
    [["county_state", col_for_outlier, "total_poverty_rate",
      "vulnerability_score", "priority_score", "z_score"]]
)

## 11. Quick Visual Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(
    data=df_clean[["total_poverty_rate", "under_5_poverty_rate", "over_75_poverty_rate"]],
    ax=axes[0]
)
axes[0].set_title("Poverty Rates by Age Group")
axes[0].set_xticklabels(["Total", "Under 5", "Over 75"], rotation=30)

sns.boxplot(x="priority_score", y="vulnerability_score", data=df_clean, ax=axes[1],
            order=["Very Low", "Low", "Medium", "High", "Very High"])
axes[1].set_title("Vulnerability Score by Priority Category")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 12. Export Results (optional)

In [ ]:
# Uncomment to write ranked tables
# top_counties.to_csv("top_vulnerable_counties.csv", index=False)
# state_counts.to_csv("states_by_very_high_counties.csv", index=False)
# outliers.to_csv("poverty_count_outliers.csv", index=False)
print("Export cells ready – uncomment as needed.")

## 13. Notes for the Next Analyst

- Document any changes to the weight vector and the rationale.  
- If you switch geography (tracts instead of counties) update the join keys and the narrative.  
- Always re-run the status-code and shape checks after changing the API endpoint.  
- Keep secrets in `.env`; never hard-code keys.  
- Sensitivity analysis around the weights is strongly recommended before final resource decisions.